In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def RK4_2(func, X0, sets):
    """
    Runge Kutta 4 solver.
    """
    n_x, dt, n_data, window, h, c, A1, A2, window_length, sigma = sets
    func_sets = (n_x, window,h, c, A1, A2, sigma)
    X  = np.zeros([n_data, X0.shape[1]])
    X[:window] = X0
    ti = (window -1)*dt
    for i in range(window, n_data - 1):
        X_seq = np.vstack([X[i - window], X[i - window  + window_length], X[i]])
        k1 = func(X_seq, ti, func_sets)
        X_seq_2 = np.vstack([(X[i - window  + 1] + X[i - window ])/2, (X[i - window + 1 + window_length] + X[i - window  + window_length])/2, X[i] + dt/2. * k1])
        k2 = func(X_seq_2, ti + dt/2., func_sets)
        X_seq_3 = np.vstack([(X[i - window  + 1] + X[i - window ])/2, (X[i - window + 1 + window_length] + X[i - window  + window_length])/2, X[i] + dt/2. * k2])
        k3 = func(X_seq_3, ti + dt/2., func_sets)
        X_seq_4 = np.vstack([X[i - window  + 1], X[i - window + 1 + window_length], X[i] + dt * k3])
        k4 = func(X_seq_4, ti + dt, func_sets)
        X[i+1] = X[i] + dt / 6. * (k1 + 2. * k2 + 2. * k3 + k4)
        ti += dt
    return X

def coupled_Lorenz_2(x, t, sets):
#     Lorenz system is time-invariant, no need to use the input t
    n_x, window, h, c, A1, A2, sigma = sets
    x_now = x[-1]
    r = np.zeros(x_now.shape)
    

    noise = np.random.normal(loc=0, scale=sigma, size=3*n_x)

    for k in range(n_x):
        xk = x_now[k*3 + 0]
        yk = x_now[k*3 + 1]
        zk = x_now[k*3 + 2]


        

        
        coupling =np.sum(A1[k] * (x[0,1::3]-yk)+ A2[k] * (x[1,1::3]-yk))


        r[k*3+0]= -10*(xk-yk +c*coupling) + noise[0+k*3]
        r[k*3+1]= 28*(1+h[k])*xk-yk - xk*zk + noise[1+k*3]
        r[k*3+2]= -8/3*zk+xk*yk + noise[2+k*3]
    return r


In [ ]:
def gen_data_norm_2time_2_3(n_x, A1, A2, n_data, window, window_length, c = 0.3, sigma_dyn = 0, sigma_obs = 0, dt = 0.01):
    h  = 2*(np.random.rand(n_x)-0.5)*0.06
    
    setting = (n_x, dt, n_data, window, h, c, A1, A2, window_length, sigma_dyn)
    #u0 = np.array([7.432487609628195, 10.02071718705213, 29.62297428638419])

    random_vector = np.random.rand(3 * n_x)
    x0 = np.tile(random_vector, (window, 1))



#     X = gen_data(setting, x0)
    X = RK4_2(coupled_Lorenz_2, x0, setting)
    
    q1 = np.percentile(X, 25, interpolation='midpoint', axis=0)
    q2 = np.percentile(X, 50, interpolation='midpoint', axis=0)
    q3 = np.percentile(X, 75, interpolation='midpoint', axis=0)

    X = (X - q2)/(q3 - q1)
    
    if sigma_obs != 0:
        noise_obs = np.random.normal(loc=0, scale=sigma_obs, size=X.shape)
        X += noise_obs
    
    return X